In [1]:
import sys

print("📦 Instalando dependências...")

!pip install torch torchvision torchaudio --quiet

!pip install numpy pandas scikit-learn joblib matplotlib seaborn --quiet
!pip install ipykernel --quiet

import torch
import sklearn
import pandas as pd
import numpy as np

print("\n✅ Instalação concluída!")
print(f"PyTorch version: {torch.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

📦 Instalando dependências...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.0 MB/s eta 0:00:00

✅ Instalação concluída!
PyTorch version: 2.9.0+cu126
Scikit-learn version: 1.6.1
Pandas version: 2.2.2
Numpy version: 2.0.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Implementação Seed

In [3]:
# === Reprodutibilidade e checkpointing (melhorado) ===
import os, time, hashlib, json, csv, random
from pathlib import Path
import joblib
import numpy as np
import torch

def set_seed(seed:int, deterministic:bool=False):
    random.seed(int(seed))
    np.random.seed(int(seed))
    try:
        torch.manual_seed(int(seed))
    except Exception:
        pass
    if torch.cuda.is_available():
        try:
            torch.cuda.manual_seed(int(seed))
            torch.cuda.manual_seed_all(int(seed))
        except Exception:
            pass
    if deterministic:
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    print(f"[seed set] {seed} (deterministic={deterministic})")

def save_checkpoint(path: str, model, optimizer=None, epoch=None, best_acc=None, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict() if hasattr(model, 'state_dict') else model,
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'best_acc': best_acc,
        'np_rng_state': np.random.get_state(),
        'py_rng_state': random.getstate(),
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'extra': extra
    }
    torch.save(ckpt, str(path))
    print(f"[checkpoint salvo] {path}")

def load_checkpoint(path: str, model, optimizer=None, map_location=None):
    map_location = map_location or ('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(path, map_location=map_location)
    if hasattr(model, 'load_state_dict') and ckpt.get('model_state_dict') is not None:
        model.load_state_dict(ckpt['model_state_dict'])
    if optimizer is not None and ckpt.get('optimizer_state_dict') is not None:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    try:
        np.random.set_state(ckpt['np_rng_state'])
        random.setstate(ckpt['py_rng_state'])
        torch.set_rng_state(ckpt['torch_rng_state'])
        if torch.cuda.is_available() and ckpt.get('cuda_rng_state') is not None:
            try:
                torch.cuda.set_rng_state_all(ckpt['cuda_rng_state'])
            except Exception:
                pass
    except Exception as e:
        print('Aviso: não foi possível restaurar completamente os estados RNG:', e)
    return ckpt, model, optimizer

def save_artifacts(prefix: str, scaler=None, label_encoder=None):
    Path(prefix).parent.mkdir(parents=True, exist_ok=True)
    if scaler is not None:
        joblib.dump(scaler, f"{prefix}_scaler.joblib")
    if label_encoder is not None:
        joblib.dump(label_encoder, f"{prefix}_labelencoder.joblib")
    print(f"[artifacts salvos] {prefix}")

def append_experiment_log(csv_path: str, row: dict):
    csv_path = Path(csv_path)
    exists = csv_path.exists()
    with open(csv_path, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)
    print(f"[log registrado] {csv_path}")


# ErgoPose Risk Classifier — Model Training

This notebook is the **third stage** of the *ErgoPose Risk Classifier* project.  
It defines, trains, and evaluates the **Artificial Neural Network (ANN)** for posture classification based on the preprocessed dataset.

### Objectives
- Load the cleaned dataset from `data/processed/`.
- Encode categorical posture labels.
- Split the data into training and testing sets.
- Define and train an ANN model for multi-class classification.
- Save the trained model and scaler to the `models/` directory.

### Input and Output
- **Input:** `data/processed/clean_postural_risk_dataset.csv`  
- **Outputs:**  
  - `models/neural_network.pkl`  
  - `models/scaler.pkl`

In [4]:
"""
Imports the necessary libraries for model definition, training, and evaluation.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import joblib
import json
from math import sqrt

import torch
from torch import Tensor, nn, optim

In [5]:
"""
Defines paths for processed data and model output directories.
"""

DATA_PATH = Path("/content/drive/MyDrive/clean_postural_risk_dataset.csv")
MODELS_PATH = Path("../models")
MODELS_PATH.mkdir(exist_ok=True)

print(f"Dataset path: {DATA_PATH}")
print(f"Models directory: {MODELS_PATH}")


Dataset path: /content/drive/MyDrive/clean_postural_risk_dataset.csv
Models directory: ../models


In [6]:
"""
Defines the number of neurons in the hidden layers by the 'Geometric Pyramid Rule'
"""

data = pd.read_csv(DATA_PATH)

X = data.drop(columns=['upperbody_label'])
y = data['upperbody_label']

input_neurons = X.shape[1]
output_neurons = 2 # Binary classification

input_output_neurons = int(sqrt(input_neurons*output_neurons))

print(f"The number of neurons in hidden layers will be: {int(input_output_neurons*0.5)} <= N <= {int(input_output_neurons*2)}")

The number of neurons in hidden layers will be: 5 <= N <= 20


## Model Architecture Proposals - rules
- Hidden layers must have beetwen **5 and 20 neurons total**. If more than 1 hidden layer is implemented, the number of neurons of both layers must **add up** to a number beetwen **5 and 20**.
- Batch size, at this initial stage, must be **default**.
- Activation function **cannot** be tanh.
- Learning rate must be $10^{-2}, 10^{-3}$ or **smaller numbers**.

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [8]:
def run_model(input_dim, hidden_dim_1, hidden_dim_2, output_dim, kf, best_acc, best_model):
    acc_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    X_t: Tensor = torch.tensor(X.values, dtype=torch.float32).to(device)

    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    y_enc = torch.tensor(y_enc, dtype=torch.float32)

    for fold, (train_index, val_index) in enumerate(kf.split(X_t)):
        X_train_t, X_val_t = X_t[train_index].to(device), X_t[val_index].to(device)
        y_train_t, y_val_t = y_enc[train_index].to(device), y_enc[val_index].to(device)

        model: MLPModel = MLPModel(input_dim, hidden_dim_1, hidden_dim_2, output_dim).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        for epoch in range(300):
            model.train()
            optimizer.zero_grad()
            outputs = model(X_train_t).squeeze()
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()
            # print(f"Epoch {epoch}: loss {loss}")
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t).squeeze()
            y_pred_t = (torch.sigmoid(val_outputs) > 0.5).long()
            y_pred = y_pred_t.cpu().numpy()
            y_true = y_val_t.cpu().numpy()

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        if acc > best_acc:
            best_acc = acc
            best_model = model

            print(f"New best model found (Acc {best_acc:.4f})")

        acc_scores.append(acc)
        precision_scores.append(prec)
        recall_scores.append(rec)
        f1_scores.append(f1)

    results = {
        "acc": np.mean(acc_scores),
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "f1": np.mean(f1_scores)
    }

    return results, best_model, best_acc

In [9]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, output_dim):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Linear(hidden_dim_2, output_dim),
        )

    def forward(self, x):
        return self.net(x)

number_tests = 10
metrics = []
best_acc = 0.0
best_model = None

for test in range(number_tests):
    seed = 42 + test
    set_seed(seed)

    num_splits = 5
    kf = KFold(
        n_splits=num_splits,
        shuffle=True,
        random_state=seed
    )

    # Hyperparameters
    input_dim: int = X.shape[1]
    hidden_dim_1 = 8
    hidden_dim_2 = 10
    output_dim = 1

    results, new_best_model, new_best_acc = run_model(input_dim, hidden_dim_1, hidden_dim_2, output_dim, kf, best_acc, best_model)

    best_acc = new_best_acc
    best_model = new_best_model

    # Salva checkpoint do melhor modelo até agora
    ckpt_dir = MODELS_PATH / f"exp_seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"best_model_seed{seed}_acc{best_acc:.4f}.pth"
    try:
        save_checkpoint(str(ckpt_path), best_model, optimizer=None, epoch=None, best_acc=best_acc,
                        extra={'seed': seed, 'test_run': test})
    except Exception as e:
        torch.save(best_model.state_dict(), str(ckpt_dir / f"best_model_seed{seed}_acc{best_acc:.4f}_statedict.pth"))

    metrics.append(results)

    print(f"Métricas da execução {test}--------------------------")
    print(f"""
    Acc: {results['acc']}
    Precision: {results['precision']}
    Recall: {results['recall']}
    F1: {results['f1']}
    """)

final_metrics = {
    "Acc": np.mean([m["acc"] for m in metrics]),
    "Precision": np.mean([m["precision"] for m in metrics]),
    "Recall": np.mean([m["recall"] for m in metrics]),
    "F1": np.mean([m["f1"] for m in metrics]),
}

print(f"Acurácia: {final_metrics['Acc']:.4f}")
print(f"Precisão: {final_metrics['Precision']:.4f}")
print(f"Revocação: {final_metrics['Recall']:.4f}")
print(f"F1: {final_metrics['F1']:.4f}")

print(f"Best model Accuracy: {best_acc:.4f}")

[seed set] 42 (deterministic=False)
New best model found (Acc 0.7883)
New best model found (Acc 0.7946)
New best model found (Acc 0.8100)
[checkpoint salvo] ../models/exp_seed_42/best_model_seed42_acc0.8100.pth
Métricas da execução 0--------------------------

    Acc: 0.787655895907576
    Precision: 0.6820907742054834
    Recall: 0.6921297109901448
    F1: 0.686731952149524
    
[seed set] 43 (deterministic=False)
[checkpoint salvo] ../models/exp_seed_43/best_model_seed43_acc0.8100.pth
Métricas da execução 1--------------------------

    Acc: 0.7893197289277931
    Precision: 0.6884542199179411
    Recall: 0.6826638842093867
    F1: 0.6849890886951284
    
[seed set] 44 (deterministic=False)
New best model found (Acc 0.8102)
[checkpoint salvo] ../models/exp_seed_44/best_model_seed44_acc0.8102.pth
Métricas da execução 2--------------------------

    Acc: 0.7759666144927411
    Precision: 0.6744988894651263
    Recall: 0.6478970947253728
    F1: 0.660678785846245
    
[seed set] 45 (

In [10]:
model_path = f"{MODELS_PATH}/modelo_{round(final_metrics['Acc'], 2)}_{hidden_dim_1}-{hidden_dim_2}.pth"
model_details_path = f"{MODELS_PATH}/modelo_{round(final_metrics['Acc'], 2)}_{hidden_dim_1}-{hidden_dim_2}.json"

model = best_model

torch.save(best_model.state_dict(), model_path)


# Refit and save LabelEncoder for inference consistency
from sklearn.preprocessing import LabelEncoder
le_final = LabelEncoder()
le_final.fit(y)
le_path = MODELS_PATH / f"labelencoder_seed{seed}.joblib"
joblib.dump(le_final, str(le_path))
print(f"LabelEncoder salvo em {le_path}")
model_details = {
    "optimizer": "Adam",
    "funcao_ativacao": "ReLU",
    "num_epochs": 300,
    "hidden_layer_1": hidden_dim_1,
    "hidden_layer_2": hidden_dim_2,
    "learning_rate": 0.0001,
    "acc": round(final_metrics['Acc'], 2),
    "precision": round(final_metrics['Precision'], 2),
    "recall": round(final_metrics['Recall'], 2),
    "f1": round(final_metrics['F1'], 2)
}

with open(model_details_path, 'w') as f:
    json.dump(model_details, f, indent=4)


print(f"Modelo salvo em {model_path}.")
print(f"Detalhes salvos em {model_details_path}")

LabelEncoder salvo em ../models/labelencoder_seed51.joblib
Modelo salvo em ../models/modelo_0.78_8-10.pth.
Detalhes salvos em ../models/modelo_0.78_8-10.json
